# SPL Loss Ratio vs NTR — All Specialty Lines, Cat ExcludedExtends the line 16 analysis to every fitted line. Same logic, looped.**Lines fit:** 16 (auto), 32 (mfg home), 71 (renters), 72 (landlord), 78 (condo), 88 (PUP), 90 (boat)**Skipped:** 62, 64, 70, 79 — Nick's `non_spl` list, not specialty.

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport statsmodels.formula.api as smffrom IPython.display import displaypd.set_option("display.width", 220)pd.set_option("display.max_columns", 40)KEYS = ["ACTMO", "ACTYR", "ALINE", "CLINE", "COMPNY", "GEOST", "NTR"]DATA = "/mnt/data/eltv-policy/eltv-notebooks-d/ssenp/SPL"# One entry per extract. If auto and property came from separate CDF pulls,# list both bases here -- they get concatenated.FILE_BASES = [    "ssenp_20260730-142446",]FLORIDA = 9# fit_mode: "ntr_gt0" -> line fit on NTR>0, NTR=0 gets standalone wavg#           "all"     -> single line through all NTR buckets# excl_st:  GEOST codes dropped before fittingLINE_SPEC = {    16: dict(label="Specialty Auto",  fit_mode="ntr_gt0", excl_st=()),    32: dict(label="Mfg Home",        fit_mode="all",     excl_st=()),    71: dict(label="Renters",         fit_mode="all",     excl_st=()),    72: dict(label="Landlord",        fit_mode="all",     excl_st=()),    78: dict(label="Condo",           fit_mode="all",     excl_st=(FLORIDA,)),    88: dict(label="PUP",             fit_mode="ntr_gt0", excl_st=()),    90: dict(label="Boat",            fit_mode="ntr_gt0", excl_st=()),}# Production fits: [slope, intercept] or [slope, intercept, ntr0_value].# TRANSCRIBED FROM ngraf's notebook -- verify against pipeline source.PROD_FITS = {    16: [-0.02294,  0.6347,  1.0057],    32: [-0.01407,  0.6206],    71: [-0.05045,  0.7259],    72: [-0.003477, 0.6060],    78: [-0.03717,  0.82678],    88: [ 0.0,      0.7571,  1.2079],    90: [-0.02329,  0.7527,  0.9611],}

## 1. Load

In [ ]:
clm = pd.concat([pd.read_csv(f"{DATA}/{b}c.csv") for b in FILE_BASES], ignore_index=True)prem = pd.concat([pd.read_csv(f"{DATA}/{b}p.csv") for b in FILE_BASES], ignore_index=True)print(f"claims  {clm.shape}  {clm.columns.tolist()}")print(f"premium {prem.shape}  {prem.columns.tolist()}")print("lines in claims :", sorted(clm.ALINE.unique()))print("lines in premium:", sorted(prem.ALINE.unique()))print("ACTYR           :", sorted(prem.ACTYR.unique()))missing = sorted(set(LINE_SPEC) - set(prem.ALINE.unique()))if missing:    print(f"\n!! configured lines absent from extract: {missing}")    print("!! re-pull before releasing -- do not fit on a partial extract")

## 2. Cat indicatorNon-blank CATCD = catastrophe. Same rule as line 16.

In [ ]:
clm["is_cat"] = clm["CATCD"].fillna("").astype(str).str.strip().ne("")clm["TOTLOSS"] = clm["CMEXP"] + clm["CMLOSS"]print("cat codes:", sorted(clm.loc[clm.is_cat, "CATCD"].str.strip().unique()))print(f"cat rows: {clm.is_cat.sum():,} of {len(clm):,}")# CATCD adds grain -- a cell with several cat events becomes several rows.# Not duplication. Confirm no TRUE duplicates on keys + CATCD.n_cells = clm[KEYS].drop_duplicates().shape[0]true_dupes = (clm.groupby(KEYS + ["CATCD"]).size() > 1).sum()print(f"rows {len(clm):,} vs cells {n_cells:,} -> grain changed: {len(clm) > n_cells}")print(f"true duplicates on keys+CATCD: {true_dupes}")

## 3. Cat share by line — which lines actually move

In [ ]:
by_line = (clm.groupby(["ALINE", "is_cat"])["TOTLOSS"].sum()              .unstack(fill_value=0)              .rename(columns={False: "loss_ex_cat", True: "loss_cat"}))by_line["loss_all"] = by_line.sum(axis=1)by_line["cat_share"] = by_line["loss_cat"] / by_line["loss_all"]by_line = by_line.loc[by_line.index.isin(LINE_SPEC)]by_line["label"] = [LINE_SPEC[i]["label"] for i in by_line.index]display(by_line[["label", "loss_all", "loss_cat", "cat_share"]].round(4))ax = by_line["cat_share"].plot(kind="bar")ax.set_title("Catastrophe share of incurred loss, by line")ax.set_xlabel("ALINE"); ax.set_ylabel("cat share")plt.tight_layout(); plt.show()

## 4. Filter cat → re-aggregate → mergeThe one structural change from Nick's notebook. Collapsing claims back to cellgrain before the merge; merging first would fan out premium rows.

In [ ]:
def claims_at_cell_grain(df, mode):    sub = {"all": df, "ex_cat": df[~df.is_cat], "cat_only": df[df.is_cat]}[mode]    return sub.groupby(KEYS, as_index=False)[["CMEXP", "CMLOSS"]].sum()def build(prem, clm, mode):    cells = prem.merge(claims_at_cell_grain(clm, mode), on=KEYS,                       how="outer", indicator=True, validate="one_to_one")    for col in ["EEXP", "EPREM", "CMEXP", "CMLOSS"]:        cells[col] = cells[col].fillna(0.0)    cells["TOTLOSS"]    = cells["CMEXP"] + cells["CMLOSS"]    cells["loss_ratio"] = np.where(cells.EPREM > 0, cells.TOTLOSS / cells.EPREM, 0.0)    cells["w"]          = cells["EPREM"]    return cellsa  = claims_at_cell_grain(clm, "all")["CMLOSS"].sum()ex = claims_at_cell_grain(clm, "ex_cat")["CMLOSS"].sum()ct = claims_at_cell_grain(clm, "cat_only")["CMLOSS"].sum()assert np.isclose(a, ex + ct, atol=0.01), "all != ex_cat + cat_only"print(f"additivity OK  |  cat = {ct/a:.1%} of CMLOSS")cells_all = build(prem, clm, "all")cells_ex  = build(prem, clm, "ex_cat")print(cells_ex["_merge"].value_counts().to_dict())# Loss on zero-premium cells gets weight 0 and vanishes from every fit.z = cells_ex.EPREM <= 0print(f"stranded on zero-premium cells: {cells_ex.loc[z,'TOTLOSS'].sum()/cells_ex.TOTLOSS.sum():.2%}")

## 5. Reconciliation — what changed in lossesBecause weight = EPREM and loss_ratio = TOTLOSS/EPREM, the weighted averageloss ratio in any bucket is just **total loss / total premium** for that bucket.The observed points are aggregate loss ratios, nothing fancier.

In [ ]:
def recon(by):    out = None    for tag, cells in [("all", cells_all), ("ex_cat", cells_ex)]:        g = (cells[cells.ALINE.isin(LINE_SPEC)]             .groupby(by).agg(premium=("EPREM","sum"), loss=("TOTLOSS","sum")))        g[f"lr_{tag}"] = g["loss"] / g["premium"]        g = g.rename(columns={"loss": f"loss_{tag}"})        out = g if out is None else out.join(g[[f"loss_{tag}", f"lr_{tag}"]])    out["lr_delta"] = out["lr_ex_cat"] - out["lr_all"]    return outrecon_line = recon(["ALINE"])recon_line.insert(0, "label", [LINE_SPEC[i]["label"] for i in recon_line.index])display(recon_line.round(4))recon_ntr = recon(["ALINE", "NTR"])

## 6. FitWLS `loss_ratio ~ NTR`, weights = earned premium. Fallback to a flat weightedaverage when the NTR coefficient is insignificant or the slope comes out positive.Note: the flat fallback averages over the **same subset used for the regression**(NTR>0 for piecewise lines). Nick's version averaged over all points even forpiecewise lines, mixing the new-business bucket back into a value meant toexclude it.

In [ ]:
def fit_line(cells, line, spec, tag):    d = cells[(cells.ALINE == line) & (cells.EPREM > 0)]    if spec["excl_st"]:        d = d[~d.GEOST.isin(spec["excl_st"])]    dg = d[d.NTR > 0] if spec["fit_mode"] == "ntr_gt0" else d    lm = smf.wls("loss_ratio ~ NTR", data=dg, weights=dg["w"]).fit()    p, raw = float(lm.pvalues["NTR"]), float(lm.params["NTR"])    flat = (p > 0.1) or (raw > 0)    slope = 0.0 if flat else raw    icept = (np.average(dg.loss_ratio, weights=dg.w) if flat             else float(lm.params["Intercept"]))    ntr0 = np.nan    if spec["fit_mode"] == "ntr_gt0":        n0 = d[d.NTR == 0]        ntr0 = np.average(n0.loss_ratio, weights=n0.w) if len(n0) else np.nan    # weighted avg loss ratio by NTR == total loss / total premium    g = d.groupby("NTR").agg(loss=("TOTLOSS","sum"), prem=("EPREM","sum"))    pts = (g["loss"] / g["prem"]).rename("lr_wavg").reset_index()    return dict(line=line, label=spec["label"], tag=tag, fit_mode=spec["fit_mode"],                slope=slope, intercept=icept, ntr0=ntr0, p_value=p,                raw_slope=raw, flat=flat, premium=d.EPREM.sum(), pts=pts)fits_all, fits_ex = {}, {}for line, spec in LINE_SPEC.items():    if line not in cells_ex.ALINE.unique():        print(f"  ! line {line} absent -- skipped")        continue    fits_all[line] = fit_line(cells_all, line, spec, "with cat")    fits_ex[line]  = fit_line(cells_ex,  line, spec, "ex cat")summary = pd.DataFrame([    {k: f[k] for k in ["line","label","fit_mode","slope","intercept","ntr0",                       "p_value","raw_slope","flat","premium"]}    for f in fits_ex.values()]).set_index("line")display(summary.round(5))

## 7. New vs production

In [ ]:
def curve(f, ntr):    y = f["intercept"] + f["slope"] * ntr    return np.where(ntr == 0, f["ntr0"], y) if not np.isnan(f["ntr0"]) else ydef prod_curve(line, ntr):    p = PROD_FITS[line]    y = p[1] + p[0] * ntr    return np.where(ntr == 0, p[2], y) if len(p) > 2 else yntr = np.arange(0, 10)rows = []for line, f in fits_ex.items():    new, old = curve(f, ntr), prod_curve(line, ntr)    prem = recon_ntr.loc[line, "premium"].reindex(ntr).fillna(0).values    rows.append({        "line": line, "label": f["label"],        "lr_prod_wavg": np.average(old, weights=prem) if prem.sum() else np.nan,        "lr_new_wavg":  np.average(new, weights=prem) if prem.sum() else np.nan,        "slope_prod": PROD_FITS[line][0], "slope_new": f["slope"],        "dollar_delta": float(((new - old) * prem).sum()),        "premium": prem.sum(),    })cmp_df = pd.DataFrame(rows).set_index("line")cmp_df["lr_pt_change"] = cmp_df["dollar_delta"] / cmp_df["premium"]display(cmp_df.round(4))print(f"total dollar impact: {cmp_df.dollar_delta.sum():,.0f}")

## 8. Per-line plots

In [ ]:
n = len(fits_ex)fig, axes = plt.subplots((n + 2) // 3, 3, figsize=(15, 4 * ((n + 2) // 3)))for ax, (line, f) in zip(np.ravel(axes), fits_ex.items()):    ax.scatter(f["pts"].NTR, f["pts"].lr_wavg, label="observed ex-cat", zorder=3)    ax.plot(ntr, prod_curve(line, ntr), ":",  color="red", label="production")    ax.plot(ntr, curve(fits_all[line], ntr), "--", label="refit, with cat")    ax.plot(ntr, curve(f, ntr), "-", label="refit, ex cat")    excl = " (excl FL)" if LINE_SPEC[line]["excl_st"] else ""    ax.set_title(f"Line {line} {f['label']}{excl}")    ax.set_xlabel("NTR"); ax.set_ylabel("avg loss ratio"); ax.legend(fontsize=7)for ax in np.ravel(axes)[n:]:    ax.axis("off")plt.tight_layout(); plt.show()

## 9. Combined — all lines, ex-cat

In [ ]:
for line, f in fits_ex.items():    plt.plot(ntr, curve(f, ntr), "-", label=f"Line {line} {f['label']}")plt.title("All specialty lines — loss ratio vs NTR, cat excluded")plt.xlabel("NTR"); plt.ylabel("average loss ratio")plt.ylim(0.2, 1.25); plt.legend(fontsize=8)plt.tight_layout(); plt.show()

## 10. Line 78 — does the Florida carve-out still earn its keep?The FL exclusion was cat-motivated: pre-filter, FL condo at NTR=0 ran a 1.93loss ratio vs 0.82 non-FL. If cat removal closed that gap, keeping the carve-outdouble-counts the fix.

In [ ]:
if 78 in fits_ex:    with_fl = fit_line(cells_ex, 78, dict(label="Condo", fit_mode="all", excl_st=()), "ex cat")    no_fl   = fits_ex[78]    print(f"  FL included: slope={with_fl['slope']:.5f}  intercept={with_fl['intercept']:.5f}")    print(f"  FL excluded: slope={no_fl['slope']:.5f}  intercept={no_fl['intercept']:.5f}")    d78 = cells_ex[(cells_ex.ALINE == 78) & (cells_ex.EPREM > 0)]    g = (d78.assign(fl=d78.GEOST.eq(FLORIDA))            .groupby(["fl", "NTR"]).agg(loss=("TOTLOSS","sum"), prem=("EPREM","sum")))    display((g["loss"] / g["prem"]).unstack(0)            .rename(columns={False: "non_FL", True: "FL"}).round(4))    print("  If FL and non-FL have converged post-cat, retire the carve-out.")

## 11. Output for the pipeline

In [ ]:
print("spl_fits = {")for line, f in sorted(fits_ex.items()):    vals = [f["slope"], f["intercept"]] + ([f["ntr0"]] if not np.isnan(f["ntr0"]) else [])    print(f"    {line}: [{', '.join(f'{v:.6f}' for v in vals)}],  # {f['label']}")print("}")print()for line, f in sorted(fits_ex.items()):    print(f'other_lr_tuple(line="{line}", slope_yr={f["slope"]:.10f}, '          f'intercept={f["intercept"]:.10f}),')

## Open items1. **NTR=9 is censored** — it means "9 or more" but enters the regression as literally 9,   and it carries the largest premium weight on several lines. Nick tried two fixes   (a polynomial weight, truncating at NTR<8) and shipped neither. Needs a decision:   drop it, downweight it, or get its true mean tenure and use that as the x-value.2. **PROD_FITS transcribed from the notebook** — verify against pipeline source before   quoting deltas.3. **Fallback rule** — production tests `p > 0.1 or slope > 0`; Nick's "Finalized Fits"   cell tests only `p > 0.1`. Which is intended?4. **Piecewise vs single-line per line** — currently carrying Nick's assignment forward.   Worth re-testing now that the loss curve has changed shape.5. **Where does the cat load get added back** downstream, since premium stays whole?6. **ACTYR coverage** — rebalancing window is 2024→present. Confirm the extract spans it.